# 03a: Build the analysis cohort

Create one auditable eligibility row per telemetry site.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p/'src'/'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from dnsp_analysis.analysis_cohort import CohortRules, build_site_eligibility, site_eligibility_path
from dnsp_analysis.config import load_config
from dnsp_analysis.db import connect
from dnsp_analysis.schemas import sql_string
pd.set_option('display.max_columns', 100); plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
CONFIG_PATH = PROJECT_ROOT/'analysis.toml'
OVERWRITE_COHORT = True
MINIMUM_P_Q_COVERAGE = 0.95
ACCEPTED_MAPPING_CONFIDENCE = ('high', 'medium')
config = load_config(CONFIG_PATH, check_inputs=False)
scope = config.scope(None, None)
rules = CohortRules(
    minimum_power_coverage=MINIMUM_P_Q_COVERAGE,
    accepted_mapping_confidence=ACCEPTED_MAPPING_CONFIDENCE,
    require_explicit_no_controlled_load=True,
    require_location=True,
)
print(rules)

In [ ]:
output = site_eligibility_path(config)
if OVERWRITE_COHORT or not output.is_file():
    summary = build_site_eligibility(config, scope, rules=rules, overwrite=OVERWRITE_COHORT)
    display(pd.DataFrame([summary]).T.rename(columns={0:'value'}))
else:
    print('Reusing existing site eligibility. Set OVERWRITE_COHORT=True to rebuild.')
con = connect(config)
eligibility = con.execute(f'SELECT * FROM read_parquet({sql_string(output)}) ORDER BY serial').fetchdf()
display(eligibility.head())

## Gate accounting

In [ ]:
gates = [c for c in eligibility.columns if c.startswith('gate_')]
gate_counts = pd.DataFrame({
    'gate': gates,
    'passed_sites': [int(eligibility[c].sum()) for c in gates],
    'failed_sites': [int((~eligibility[c]).sum()) for c in gates],
})
display(gate_counts)
display(eligibility.groupby(['analysis_cohort','controlled_load_status','phase_mapping_confidence'], dropna=False).size().rename('n_sites').reset_index())
display(eligibility['exclusion_reasons'].value_counts(dropna=False).rename_axis('exclusion_reasons').reset_index(name='n_sites').head(25))

In [ ]:
coverage = eligibility[['minimum_active_power_coverage','minimum_reactive_power_coverage','minimum_joint_power_coverage']]
display(coverage.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
fig, ax = plt.subplots(figsize=(9,4))
eligibility.minimum_joint_power_coverage.hist(ax=ax, bins=20)
ax.axvline(MINIMUM_P_Q_COVERAGE, color='red', linestyle='--', label='eligibility threshold')
ax.set(xlabel='Minimum P/Q coverage across inferred DER phases', ylabel='Sites')
ax.legend(); plt.show()

## Final cohort gate

In [ ]:
eligible = eligibility.loc[eligibility.eligible_for_irradiance_assessment].copy()
display(eligible[['serial','inferred_der_phases','phase_mapping_method','phase_mapping_confidence','minimum_joint_power_coverage','controlled_load_status','sub_lat','sub_long','solar_capacity_kw']].head(30))
print('Eligible for irradiance assessment:', len(eligible))
assert eligibility.serial.is_unique
assert eligible.analysis_cohort.eq('solar_only').all()
assert (~eligible.has_battery).all()
assert eligible.controlled_load_status.eq('no').all()
assert eligible.minimum_joint_power_coverage.ge(MINIMUM_P_Q_COVERAGE).all()
assert eligible[['sub_lat','sub_long']].notna().all().all()
con.close()
print('Analysis cohort gate passed.')